# RF Anomaly Detection - Interactive Exploration

This notebook provides an interactive environment for exploring the RF anomaly detection system.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.synthetic import SyntheticRFGenerator, Modulation, AnomalyType
from src.data.snr_estimation import estimate_snr
from src.data.datasets import RFDataset
from src.models.snr_encoder import create_model, SNRConditionedVAE
from src.utils.config import load_config
from src.utils.visualization import (
    plot_signals, plot_constellation, plot_reconstruction,
    plot_latent_space, plot_detection_curves
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Synthetic Signal Generation

Explore the synthetic RF signal generator.

In [ ]:
# Create generator
generator = SyntheticRFGenerator(
    sequence_length=1024,
    sample_rate=1e6,
    seed=42
)

print("Available modulations:", [m.value for m in Modulation])
print("Available anomaly types:", [a.value for a in AnomalyType])

In [ ]:
# Generate and visualize normal signals with different modulations
fig, axes = plt.subplots(4, 2, figsize=(14, 12))

for i, mod in enumerate(Modulation):
    iq, meta = generator.generate_normal_signal(modulation=mod.value, snr_db=20)
    
    axes[i, 0].plot(iq[0, :200], label='I')
    axes[i, 0].plot(iq[1, :200], label='Q')
    axes[i, 0].set_title(f'{mod.value.upper()} - Time Domain')
    axes[i, 0].legend()
    
    axes[i, 1].scatter(iq[0, ::10], iq[1, ::10], alpha=0.5, s=5)
    axes[i, 1].set_title(f'{mod.value.upper()} - Constellation')
    axes[i, 1].set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Generate and visualize anomalous signals
fig, axes = plt.subplots(len(AnomalyType), 2, figsize=(14, 15))

for i, anom in enumerate(AnomalyType):
    iq, meta = generator.generate_anomaly(anomaly_type=anom.value, snr_db=20)
    
    axes[i, 0].plot(iq[0], alpha=0.7)
    axes[i, 0].set_title(f'{anom.value.replace("_", " ").title()} - I Channel')
    
    axes[i, 1].scatter(iq[0, ::5], iq[1, ::5], alpha=0.3, s=3)
    axes[i, 1].set_title(f'{anom.value.replace("_", " ").title()} - Constellation')
    axes[i, 1].set_aspect('equal')

plt.tight_layout()
plt.show()

## 2. SNR Effects

Visualize how SNR affects signal quality.

In [ ]:
# Generate QPSK signals at different SNR levels
snr_levels = [-5, 0, 5, 10, 15, 20, 30]

fig, axes = plt.subplots(2, len(snr_levels), figsize=(20, 6))

for i, snr in enumerate(snr_levels):
    iq, meta = generator.generate_normal_signal(modulation='qpsk', snr_db=snr)
    estimated_snr = estimate_snr(iq)
    
    axes[0, i].plot(iq[0, :100], linewidth=0.5)
    axes[0, i].set_title(f'SNR={snr}dB\n(Est: {estimated_snr:.1f}dB)')
    axes[0, i].set_ylim(-1.5, 1.5)
    
    axes[1, i].scatter(iq[0, ::10], iq[1, ::10], alpha=0.5, s=3)
    axes[1, i].set_xlim(-1.5, 1.5)
    axes[1, i].set_ylim(-1.5, 1.5)
    axes[1, i].set_aspect('equal')

axes[0, 0].set_ylabel('Time Domain')
axes[1, 0].set_ylabel('Constellation')
plt.tight_layout()
plt.show()

## 3. Model Architecture

Explore the model architecture and test forward passes.

In [ ]:
# Load configuration
config = load_config('../configs/default.yaml')
print(f"Model type: {config.model.type}")
print(f"Latent dim: {config.model.latent_dim}")
print(f"Hidden channels: {config.model.hidden_channels}")

In [ ]:
# Create model
model = create_model(config)
print(model)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {num_params:,}")

In [ ]:
# Test forward pass
x = torch.randn(4, 2, config.data.sequence_length)
snr = torch.rand(4)

model.eval()
with torch.no_grad():
    x_recon, mu, logvar, z = model(x, snr)

print(f"Input shape: {x.shape}")
print(f"Reconstruction shape: {x_recon.shape}")
print(f"Latent mean shape: {mu.shape}")
print(f"Latent log-var shape: {logvar.shape}")

## 4. Dataset and DataLoader

Create and explore datasets.

In [ ]:
# Create dataset
dataset = RFDataset.from_generator(
    generator=generator,
    num_samples=1000,
    anomaly_ratio=0.1,
    snr_range=(-5, 30),
)

print(f"Dataset size: {len(dataset)}")

# Get a sample
sample = dataset[0]
print(f"\nSample keys: {sample.keys()}")
print(f"IQ shape: {sample['iq'].shape}")
print(f"Label: {sample['label']}")
print(f"SNR (normalized): {sample['snr']:.3f}")
print(f"SNR (dB): {sample['snr_db']:.1f}")

In [ ]:
# Analyze dataset statistics
labels = [dataset[i]['label'].item() for i in range(len(dataset))]
snrs = [dataset[i]['snr_db'].item() for i in range(len(dataset))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
axes[0].bar(['Normal', 'Anomaly'], 
            [labels.count(0), labels.count(1)])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')

# SNR distribution
axes[1].hist(snrs, bins=20)
axes[1].set_title('SNR Distribution')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Reconstruction Visualization (Untrained Model)

Visualize reconstructions from an untrained model to see the baseline.

In [ ]:
# Get some samples
normal_idx = next(i for i in range(len(dataset)) if dataset[i]['label'] == 0)
anomaly_idx = next(i for i in range(len(dataset)) if dataset[i]['label'] == 1)

normal_sample = dataset[normal_idx]
anomaly_sample = dataset[anomaly_idx]

# Forward pass
model.eval()
with torch.no_grad():
    normal_recon, _, _, _ = model(
        normal_sample['iq'].unsqueeze(0),
        normal_sample['snr'].unsqueeze(0)
    )
    anomaly_recon, _, _, _ = model(
        anomaly_sample['iq'].unsqueeze(0),
        anomaly_sample['snr'].unsqueeze(0)
    )

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (original, recon, title) in enumerate([
    (normal_sample['iq'], normal_recon[0], 'Normal'),
    (anomaly_sample['iq'], anomaly_recon[0], 'Anomaly')
]):
    axes[i, 0].plot(original[0, :200].numpy(), label='Original I', alpha=0.7)
    axes[i, 0].plot(recon[0, :200].numpy(), label='Reconstructed I', alpha=0.7)
    axes[i, 0].set_title(f'{title} - I Channel (Untrained)')
    axes[i, 0].legend()
    
    error = ((original - recon) ** 2).mean()
    axes[i, 1].plot((original[0] - recon[0]).numpy()[:200], color='red', alpha=0.7)
    axes[i, 1].set_title(f'{title} - Error (MSE: {error:.4f})')

plt.tight_layout()
plt.show()

## 6. Training Loop Example

A minimal training loop to see the model learning.

In [ ]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Create smaller dataset for quick demo
demo_generator = SyntheticRFGenerator(sequence_length=256, seed=42)
demo_dataset = RFDataset.from_generator(
    generator=demo_generator,
    num_samples=500,
    anomaly_ratio=0.0,  # Train on normal only
)
demo_loader = DataLoader(demo_dataset, batch_size=32, shuffle=True)

# Create smaller model for demo
demo_model = SNRConditionedVAE(
    latent_dim=16,
    sequence_length=256,
    hidden_channels=[16, 32, 64],
)

optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3)

# Quick training
losses = []
demo_model.train()

for epoch in range(10):
    epoch_loss = 0
    for batch in demo_loader:
        iq = batch['iq']
        snr = batch['snr']
        
        optimizer.zero_grad()
        x_recon, mu, logvar, _ = demo_model(iq, snr)
        loss, _, _ = demo_model.loss(iq, x_recon, mu, logvar)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(demo_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# Visualize learned reconstructions
demo_model.eval()

test_sample = demo_dataset[0]
with torch.no_grad():
    test_recon, _, _, _ = demo_model(
        test_sample['iq'].unsqueeze(0),
        test_sample['snr'].unsqueeze(0)
    )

fig = plot_reconstruction(
    test_sample['iq'],
    test_recon[0],
    figsize=(12, 5)
)
fig.suptitle('Reconstruction After Training', y=1.02)
plt.show()

## 7. Next Steps

To fully train and evaluate the model:

1. **Train baseline model:**
   ```bash
   python experiments/train_baseline.py --config configs/default.yaml
   ```

2. **Evaluate model:**
   ```bash
   python experiments/evaluate.py --checkpoint checkpoints/best_model.pt --save-plots
   ```

3. **Compare learning methods:**
   ```bash
   python experiments/compare_learning.py --baseline-checkpoint checkpoints/best_model.pt
   ```